# Filters and Transitions in VideoDB Editor
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/video-db/videodb-cookbook/blob/main/editor/feature/filters_and_transitions.ipynb)

In this notebook, we'll explore how to apply **filters** and **transitions** to video clips using VideoDB Editor.

## What Are Filters?

Filters are color treatments that change the mood and visual appearance of your clips. Think of them as Instagram-style effects that you can apply programmatically—greyscale for a classic feel, blur for artistic effects, or negative for surreal visuals.

## What Are Transitions?

Transitions control how clips appear and disappear on the timeline. Instead of abrupt cuts, we can use smooth fade-in and fade-out effects that add professional polish to our videos.

## Why This Matters

Without filters and transitions, video edits can feel jarring and unpolished. These effects help us:
- Set the mood and tone of our content
- Create smooth scene changes
- Add cinematic quality without needing a GUI editor
- Build professional-looking videos entirely with code

## What We'll Cover

In this notebook, we will:
1. Apply all 8 available filters (greyscale, blur, contrast, darken, lighten, boost, muted, negative)
2. Demonstrate fade-in and fade-out transitions
3. Control transition timing and duration
4. Combine filters with transitions for advanced effects
5. Show sequential clips with smooth transitions between scenes

Let's get started!

---

## 📦 Step 1: Installing VideoDB Editor SDK

First, we need to install the VideoDB Python SDK with Editor support. Run the cell below to install it quietly.

In [ ]:
!pip -q install videodb


---

## 📦 Step 2: Connecting to VideoDB

Now we'll establish a connection to VideoDB using your API key. The key will be requested securely and won't be visible in the notebook output.

In [ ]:
import videodb
import os
from getpass import getpass

api_key = getpass("Please enter your VideoDB API Key: ")

os.environ["VIDEO_DB_API_KEY"] = api_key

conn = videodb.connect()

coll = conn.get_collection()

print("✅ Connected to VideoDB successfully!")

---

## 📦 Step 3: Uploading Video Assets

For this notebook, we'll upload multiple videos to demonstrate different filters and transitions. We'll upload:
- A **main video** with good color variety (landscapes work great for showing filter effects)
- A **second video** for transition demonstrations between clips

Run the cell below to upload the first video from YouTube.

In [ ]:
# Upload first video (use a video with vibrant colors - landscapes, nature scenes work well)
video1 = coll.upload(url="https://www.youtube.com/watch?v=wU0PYcCsL6o")
print(f"✅ Uploaded video 1: {video1.id}")

# If you've already uploaded this video, use this instead:
# video1 = coll.get_video("your_video_id_here")

Now let's upload a second video for demonstrating transitions between different clips.

In [ ]:
# Upload second video (any video with different content from the first)
video2 = coll.upload(url="https://www.youtube.com/watch?v=LejnTJL173Y")
print(f"✅ Uploaded video 2: {video2.id}")

# If you've already uploaded this video, use this instead:
# video2 = coll.get_video("your_video_id_here")

---

## 📦 Step 4: Importing Editor Components

Let's import all the Editor components we'll need for this notebook: Timeline, Track, Clip, VideoAsset, Filter, Transition, and the play_stream function to preview our results.

In [ ]:
from videodb import play_stream
from videodb.editor import Timeline, Track, Clip, VideoAsset, Filter, Transition, ImageAsset

print("✅ Editor components imported successfully!")

---

## 📦 Step 5: Understanding Filters

Filters are **clip-level effects** that change the visual appearance of your content. VideoDB Editor provides 8 different filters:

| Filter | Effect |
|--------|--------|
| `Filter.greyscale` | Removes all color, creating a black-and-white look |
| `Filter.blur` | Blurs the scene for artistic or privacy effects |
| `Filter.contrast` | Increases contrast, making darks darker and lights lighter |
| `Filter.darken` | Darkens the entire scene |
| `Filter.lighten` | Lightens the entire scene |
| `Filter.boost` | Boosts both contrast and saturation for vibrant colors |
| `Filter.muted` | Reduces saturation and contrast for a subdued look |
| `Filter.negative` | Inverts colors for a surreal, negative effect |

Filters are applied at the **Clip level**, not the Asset level. This means the same video asset can be used with different filters in different clips.

---

## 📦 Step 6: Applying a Basic Filter (Greyscale)

Let's start with the most common filter: greyscale. We'll apply it to our first video for 10 seconds.

Notice how we pass `filter=Filter.greyscale` to the Clip (not the VideoAsset). The timeline uses a neutral gray background (`#2B2B2B`) to focus on the content.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

clip = Clip(
    asset=VideoAsset(id=video1.id),
    duration=10,
    filter=Filter.greyscale
)

track = Track()
track.add_clip(0, clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Stream URL: {stream_url}")
play_stream(stream_url)

---

## 📦 Step 7: Showcasing All 8 Filters Sequentially

Now let's create a single video that shows all 8 filters one after another. Each filter will be displayed for 5 seconds, giving us enough time to see the effect clearly.

This is a great way to compare filters and understand their visual impact on the same source material.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

track = Track()

# Create clips with each filter, 5 seconds each
filters_list = [
    Filter.greyscale,
    Filter.blur,
    Filter.contrast,
    Filter.darken,
    Filter.lighten,
    Filter.boost,
    Filter.muted,
    Filter.negative
]

start_time = 0
for filter_type in filters_list:
    clip = Clip(
        asset=VideoAsset(id=video1.id, start=10),
        duration=5,
        filter=filter_type
    )
    track.add_clip(start_time, clip)
    start_time += 5

timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Stream URL: {stream_url}")
play_stream(stream_url)

**Tip:** In the video above, you'll see each filter applied for 5 seconds. Notice how:
- **Greyscale** creates a timeless, classic look
- **Blur** softens the scene (great for backgrounds or privacy)
- **Contrast** makes the image more dramatic
- **Darken** and **Lighten** adjust brightness
- **Boost** makes colors pop
- **Muted** creates a subtle, understated feel
- **Negative** inverts everything for artistic effects

---

## 📦 Step 8: Combining Filters with Other Clip Properties

Filters work seamlessly with other clip properties like `scale` and `opacity`. Let's apply a greyscale filter combined with reduced opacity and slight scaling to create a faded, vintage look.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

clip = Clip(
    asset=VideoAsset(id=video1.id),
    duration=10,
    filter=Filter.greyscale,
    scale=1.2,
    opacity=0.7
)

track = Track()
track.add_clip(0, clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Stream URL: {stream_url}")
play_stream(stream_url)

**Note:** This combination of greyscale + opacity + scale creates a dreamy, vintage aesthetic. The gray background shows through the semi-transparent clip, giving it a faded look.

---

## 📦 Step 9: Understanding Transitions

Transitions control how clips **appear** and **disappear** on the timeline. Instead of abrupt cuts, we can use smooth animations.

VideoDB Editor currently supports **fade** transitions with two parameters:
- **in_**: Controls the entrance animation (note the underscore, since `in` is a Python keyword)
- **out**: Controls the exit animation
- **duration**: How long the transition lasts (in seconds)

Transitions are applied at the **Clip level** using the `Transition` object.

---

## 📦 Step 10: Basic Fade In and Fade Out

Let's apply both fade-in and fade-out transitions to a clip. We'll use a 2-second duration for each transition, which provides a smooth, professional look.

Notice that we start the clip at 2 seconds on the timeline (`track.add_clip(2, clip)`) to see the fade-in effect clearly.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

clip = Clip(
    asset=VideoAsset(id=video1.id),
    duration=10,
    transition=Transition(in_="fade", out="fade", duration=2)
)

track = Track()
track.add_clip(2, clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Stream URL: {stream_url}")
play_stream(stream_url)

**Tip:** The clip will:
1. Start at 2 seconds on the timeline
2. Fade in from transparent to full opacity over 2 seconds
3. Play normally for 6 seconds (10 - 2 - 2)
4. Fade out to transparent over the final 2 seconds

---

## 📦 Step 11: Transition Variations (Only In or Only Out)

We don't always need both transitions. Let's try using only a fade-in transition without a fade-out.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

clip = Clip(
    asset=VideoAsset(id=video1.id),
    duration=10,
    transition=Transition(in_="fade", duration=2)
)

track = Track()
track.add_clip(2, clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Stream URL: {stream_url}")
play_stream(stream_url)

Now let's try the opposite—only a fade-out transition.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

clip = Clip(
    asset=VideoAsset(id=video1.id),
    duration=10,
    transition=Transition(out="fade", duration=2)
)

track = Track()
track.add_clip(0, clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Stream URL: {stream_url}")
play_stream(stream_url)

---

## 📦 Step 12: Transitions Work with All Asset Types

Transitions aren't limited to video clips. Let's upload an image and apply a fade transition to it.

In [ ]:
# Upload an image
image = coll.upload(url="https://images.unsplash.com/photo-1506905925346-21bda4d32df4")
print(f"✅ Uploaded image: {image.id}")

# If you've already uploaded this image:
# image = coll.get_image("your_image_id_here")

Now let's apply fade transitions to the image clip.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

clip = Clip(
    asset=ImageAsset(id=image.id),
    duration=8,
    transition=Transition(in_="fade", out="fade", duration=2)
)

track = Track()
track.add_clip(0, clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Stream URL: {stream_url}")
play_stream(stream_url)

---

## 📦 Step 13: Sequential Clips with Smooth Transitions

One of the most powerful uses of transitions is creating smooth scene changes between different video clips. Let's create a sequence using our two uploaded videos with fade transitions between them.

This creates a professional-looking video montage without any jarring cuts.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

track = Track()

# First clip with fade-out
clip1 = Clip(
    asset=VideoAsset(id=video1.id),
    duration=8,
    transition=Transition(out="fade", duration=2)
)

# Second clip with fade-in and fade-out
clip2 = Clip(
    asset=VideoAsset(id=video2.id),
    duration=8,
    transition=Transition(in_="fade", out="fade", duration=2)
)

# Third clip with fade-in
clip3 = Clip(
    asset=VideoAsset(id=video1.id, start=20),
    duration=8,
    transition=Transition(in_="fade", duration=2)
)

track.add_clip(0, clip1)
track.add_clip(8, clip2)
track.add_clip(16, clip3)

timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Stream URL: {stream_url}")
play_stream(stream_url)

**Note:** This creates a 24-second video with three scenes:
- **0-8s:** First video fades out at the end
- **8-16s:** Second video fades in at the start and fades out at the end
- **16-24s:** First video (different segment) fades in at the start

The transitions create smooth, professional scene changes.

---

## 📦 Step 14: Combining Filters and Transitions

Now for the best part—we can combine filters and transitions on the same clip. Let's create a dramatic effect with greyscale filter and fade transitions.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

clip = Clip(
    asset=VideoAsset(id=video1.id),
    duration=12,
    filter=Filter.greyscale,
    transition=Transition(in_="fade", out="fade", duration=2)
)

track = Track()
track.add_clip(2, clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Stream URL: {stream_url}")
play_stream(stream_url)

**Tip:** This creates a cinematic black-and-white sequence that fades in and out smoothly. Perfect for dramatic flashbacks or artistic segments!

---

## 📦 Step 15: Advanced Effect Stacking

Let's push the limits by combining multiple effects: filter, transition, scale, and opacity. This creates a sophisticated, layered visual effect.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

clip = Clip(
    asset=VideoAsset(id=video1.id),
    duration=10,
    filter=Filter.muted,
    transition=Transition(in_="fade", out="fade", duration=2),
    scale=1.3,
    opacity=0.8
)

track = Track()
track.add_clip(2, clip)
timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Stream URL: {stream_url}")
play_stream(stream_url)

**What's happening here:**
- **Muted filter** reduces saturation and contrast for a subtle, dreamy look
- **Fade transitions** (2 seconds each) create smooth entry and exit
- **Scale 1.3** zooms in slightly, creating a Ken Burns effect
- **Opacity 0.8** makes the clip semi-transparent, revealing the gray background

This combination creates a sophisticated, artistic effect perfect for montages or memory sequences.

---

## 📦 Step 16: Creating a Filter Showcase with Different Transitions

Let's create a more complex example: multiple clips with different filters, each with its own transition timing. This demonstrates how filters and transitions work together to create varied visual experiences.

In [ ]:
timeline = Timeline(conn)
timeline.background = "#2B2B2B"

track = Track()

# Greyscale with long fade
clip1 = Clip(
    asset=VideoAsset(id=video1.id, start=5),
    duration=7,
    filter=Filter.greyscale,
    transition=Transition(in_="fade", out="fade", duration=2)
)

# Boost with quick fade
clip2 = Clip(
    asset=VideoAsset(id=video2.id, start=10),
    duration=7,
    filter=Filter.boost,
    transition=Transition(in_="fade", out="fade", duration=1)
)

# Negative with long fade
clip3 = Clip(
    asset=VideoAsset(id=video1.id, start=30),
    duration=7,
    filter=Filter.negative,
    transition=Transition(in_="fade", out="fade", duration=2)
)

track.add_clip(0, clip1)
track.add_clip(7, clip2)
track.add_clip(14, clip3)

timeline.add_track(track)

stream_url = timeline.generate_stream()
print(f"Stream URL: {stream_url}")
play_stream(stream_url)

**Notice:** Each clip has:
- A different **filter** (greyscale, boost, negative)
- A different **transition duration** (2s, 1s, 2s)
- A different **source video segment**

This creates a dynamic, visually interesting sequence where each scene has its own character.

---

## 🎬 Wrap-Up: What We've Learned

Congratulations! We've covered the complete toolkit for filters and transitions in VideoDB Editor.

### Filters

We explored all 8 available filters and their effects:
- **Filter.greyscale** - Classic black-and-white look
- **Filter.blur** - Artistic softening or privacy effect
- **Filter.contrast** - Dramatic darks and lights
- **Filter.darken** - Reduce overall brightness
- **Filter.lighten** - Increase overall brightness
- **Filter.boost** - Vibrant colors with enhanced contrast
- **Filter.muted** - Subtle, understated colors
- **Filter.negative** - Surreal color inversion

### Transitions

We learned how to use fade transitions:
- **Fade in** (`in_="fade"`) - Smooth appearance
- **Fade out** (`out="fade"`) - Smooth disappearance
- **Duration control** - Customize transition length (recommended: 2 seconds)
- **Independent application** - Use fade-in, fade-out, or both

### Key Insights

1. **Clip-level effects**: Both filters and transitions are applied to Clips, not Assets. This means the same video can have different effects in different contexts.

2. **Combinations work seamlessly**: Filters, transitions, scale, opacity, and position can all be combined on a single clip for sophisticated effects.

3. **Professional polish without GUI**: We can create cinematic, polished videos entirely through code—no timeline scrubbing or effect panels needed.

4. **Works with all assets**: Filters and transitions apply to VideoAssets, ImageAssets, and even TextAssets.

### Practical Use Cases

- **Social media content**: Add greyscale or boost filters for consistent brand aesthetics
- **Video montages**: Use transitions for smooth scene changes
- **Artistic sequences**: Combine filters, transitions, and opacity for creative effects
- **Flashback scenes**: Greyscale + fade transitions for temporal storytelling
- **Slideshow presentations**: Image clips with fade transitions

### What's Next?

Now that we understand filters and transitions, we can:
- Experiment with different filter combinations on the same video
- Create custom transition timing patterns for rhythmic editing
- Combine these effects with layered tracks for complex compositions
- Build reusable templates for common video styles

Happy editing! 🎥✨